<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH12/CH12_NB02_arithmetic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

# Rearchitecting LLMs
## Structural techniques for efficient models

### Chapter 12: Capstone I — Replacing one agent call with a specialized SLM in production — NB02: Padding and KV Cache Arithmetic

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
Colab Environment: CPU, no GPU required

_____
This notebook accompanies sections 12.7 and 12.9. It uses small Python calculations to show how hardware-aligned widths can introduce padding waste, how attention-layer count affects KV cache size, and how to estimate a break-even point against API costs.

The cost examples in section 12.9 are illustrative stand-ins, not measured traces. The notebook is fully self-contained and requires no model weights or GPU.

In [ ]:
def padding_report(target_width, multiple=128):
    served = ((target_width + multiple - 1) // multiple) * multiple   #A
    return served, served - target_width

def snap_to_aligned(target_width, multiple=128):
    return (target_width // multiple) * multiple                      #B

raw = int(8192 * 0.80)                 # 6553, the naive 20% cut
served, waste = padding_report(raw)    # kernel actually runs 6656 wide
aligned = snap_to_aligned(raw)         # prune to 6528 instead, zero waste
print(f"naive target {raw}: served {served}, wasted columns {waste}")
print(f"aligned target {aligned}: served {aligned}, wasted columns 0")

naive target 6553: served 6656, wasted columns 103
aligned target 6528: served 6528, wasted columns 0


In [ ]:
def kv_cache_bytes(L_attn, H_kv, d_head, seq_len, batch, dtype_bytes=2):
    return 2 * L_attn * H_kv * d_head * seq_len * batch * dtype_bytes

base = kv_cache_bytes(L_attn=28, H_kv=8, d_head=128, seq_len=2048, batch=64)
slim = kv_cache_bytes(L_attn=16, H_kv=8, d_head=128, seq_len=2048, batch=64)
print(f"cache after pruning is {slim / base:.1%} of the original")

cache after pruning is 57.1% of the original


In [ ]:
small = kv_cache_bytes(L_attn=16, H_kv=8, d_head=128, seq_len=300, batch=8)
print(f"{small / 1e6:.2f} MB ({small / 2**20:.0f} MiB)")

157.29 MB (150 MiB)


## Section 12.9 — Proving it pays off

Cost model and latency report. The traces below are constructed stand-ins
(illustrative), not measurements.

In [ ]:
def breakeven_requests_per_hour(gpu_cost_per_hour, api_cost_per_request):
    return gpu_cost_per_hour / api_cost_per_request  #1

def slm_cost_per_request(gpu_cost_per_hour, requests_per_hour,
                         api_cost, escalation):
    fixed = gpu_cost_per_hour / requests_per_hour  #2
    return fixed + escalation * api_cost  #3

api_cost = 0.004
be = breakeven_requests_per_hour(gpu_cost_per_hour=2.50,
                                 api_cost_per_request=api_cost)
slm = slm_cost_per_request(
    gpu_cost_per_hour=2.50, requests_per_hour=2950,
    api_cost=api_cost, escalation=0.042)
saving = 1 - slm / api_cost
print(f"break-even at {be:.0f} requests/hour")
print(f"SLM cost per request ${slm:.6f} ({saving:.1%} below the API)")

break-even at 625 requests/hour
SLM cost per request $0.001015 (74.6% below the API)
